### Overview

In this notebook, we will fetch the active fire detections for our AOI from [NASA FIRMS](https://firms.modaps.eosdis.nasa.gov/) (Fire Information for Resource Management System). FIRMS distributes near real-time (NRT) detections from the MODIS and VIIRS sensors, usually within 3 hours of a satellite pass.

### Overview of the Task

We will use the [FIRMS Area API](https://firms.modaps.eosdis.nasa.gov/api/area/) to download all NRT detections within the bounding box of our AOI, keep the ones from the past 24 hours that fall inside the AOI polygon and save them as `fires.geojson` in the `data/` folder.

### Setup

Determine our runtime environment.

In [ ]:
import os

if 'COLAB_RELEASE_TAG' in os.environ:
    environment = 'colab'
    if os.environ.get('VERTEX_PRODUCT') == 'COLAB_ENTERPRISE':
        environment = 'colab_enterprise'
else:
    environment = 'local'

print(f'Environment: {environment}')

If we are on Google Colab, install the required packages. Local runtimes are expected to have the packages already installed.

In [ ]:
%%capture
if environment in ['colab', 'colab_enterprise']:
  !pip install geopandas

Import packages.

In [ ]:
import geopandas as gpd
import pandas as pd
import os

In [ ]:
data_folder = 'data'

### Get the FIRMS API Key

The FIRMS API requires a free MAP_KEY. Request one at the [FIRMS MAP_KEY page](https://firms.modaps.eosdis.nasa.gov/api/map_key/).

Keep the key out of your code. We store it in an environment variable named `FIRMS_MAP_KEY` and read it from the notebook.

- On your computer: Create a file named `.env` in the project folder with a single line `FIRMS_MAP_KEY=your_key`. VS Code [loads this file](https://code.visualstudio.com/docs/python/environments#_environment-variables) into the notebook kernel. Restart the kernel after creating or editing the file. The `.gitignore` file in this project lists `.env`, so your key is never committed to Git.
- On Google Colab: Add a secret named `FIRMS_MAP_KEY` from the *Secrets* panel (the key icon in the left sidebar).

In [ ]:
if environment in ['colab', 'colab_enterprise']:
    from google.colab import userdata
    map_key = userdata.get('FIRMS_MAP_KEY')
else:
    map_key = os.environ['FIRMS_MAP_KEY']

### Read the AOI

We read the AOI saved in the previous notebook. The FIRMS Area API takes a bounding box in the format `west,south,east,north`, which we get from [total_bounds](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.total_bounds.html).

In [ ]:
aoi_path = os.path.join(data_folder, 'aoi.geojson')
aoi_gdf = gpd.read_file(aoi_path)

west, south, east, north = aoi_gdf.total_bounds
bbox = f'{west},{south},{east},{north}'
bbox

### Fetch the Fire Detections

FIRMS provides NRT data for each sensor as a separate source. We fetch detections from all four sensors: MODIS (Terra and Aqua) and VIIRS (Suomi NPP, NOAA-20 and NOAA-21).

The API returns detections for full days in UTC. We request the last 2 days so we can cover the past 24 hours from now in the next step.

In [ ]:
sources = ['MODIS_NRT', 'VIIRS_SNPP_NRT', 'VIIRS_NOAA20_NRT', 'VIIRS_NOAA21_NRT']
day_range = 2

dfs = []
for source in sources:
    url = (
        'https://firms.modaps.eosdis.nasa.gov/api/area/csv/'
        f'{map_key}/{source}/{bbox}/{day_range}'
    )
    df = pd.read_csv(url)
    print(f'{source}: {len(df)} detections')
    dfs.append(df)

fires_df = pd.concat(dfs, ignore_index=True)
fires_df

### Keep the Past 24 Hours

Each detection has an acquisition date (`acq_date`) and time (`acq_time`) in UTC. The time is stored as a number in HHMM format, so 0545 is read as 545. We pad it to 4 digits and combine both columns into a single timestamp using [pd.to_datetime()](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html).

In [ ]:
acq_time = fires_df['acq_time'].astype(str).str.zfill(4)
fires_df['acq_datetime'] = pd.to_datetime(
    fires_df['acq_date'] + ' ' + acq_time, format='%Y-%m-%d %H%M', utc=True)

Now we keep only the detections from the past 24 hours.

In [ ]:
now = pd.Timestamp.now(tz='UTC')
last_24h = fires_df[fires_df['acq_datetime'] >= now - pd.Timedelta(hours=24)]
print(f'Detections in the past 24 hours: {len(last_24h)}')

### Clip to the AOI

The detections cover the whole bounding box. We convert them to a GeoDataFrame and use [clip()](https://geopandas.org/en/stable/docs/reference/api/geopandas.clip.html) to keep only the points inside the AOI polygon.

We also keep only the columns the dashboard needs. The `confidence` column has numbers (0-100) for MODIS and letters (l, n, h for low, nominal, high) for VIIRS, so we store it as text.

In [ ]:
columns = ['acq_datetime', 'satellite', 'instrument', 'confidence', 'frp', 'daynight']

fires_gdf = gpd.GeoDataFrame(
    last_24h[columns],
    geometry=gpd.points_from_xy(last_24h['longitude'], last_24h['latitude']),
    crs='EPSG:4326')

fires_gdf['confidence'] = fires_gdf['confidence'].astype(str)
fires_gdf['acq_datetime'] = fires_gdf['acq_datetime'].dt.strftime('%Y-%m-%dT%H:%MZ')

fires_gdf = gpd.clip(fires_gdf, aoi_gdf)
print(f'Detections in the AOI: {len(fires_gdf)}')

We can visualize the detections along with the AOI. If there are no fires in the past 24 hours, only the AOI is shown.

In [ ]:
m = aoi_gdf.explore(style_kwds={'fill': False})
if len(fires_gdf) > 0:
    fires_gdf.explore(m=m, color='red')
m

### Save the Results

We save the detections as a GeoJSON file. The dashboard reads this file and shows the total count.

In [ ]:
output_path = os.path.join(data_folder, 'fires.geojson')
fires_gdf.to_file(output_path)
print(f'Saved to {output_path}')